# Ноутбук для запуска на колабе

In [3]:
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("Вставь HF-токен: ").strip()

In [4]:
from datasets import load_dataset
from tqdm.auto import tqdm
from pathlib import Path

data_dir = Path("/Users/hadhad/HSE/NLP/HW2/data")
ds = load_dataset(
    "SlavaYus/tinysms_qa_12m",
    token=os.environ["HF_TOKEN"],
    cache_dir=str(data_dir / "cache")
 )

/Users/hadhad/HSE/NLP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
print(ds["train"][0])
print(len(ds["train"]))
print(len(ds["test"]))

{'question': 'Mark has 10 crayons. He gives 2 crayons to his younger sister and loses another 4 while he was playing. How many crayons does Mark have left?', 'code': 'def solve() -> int:\n    crayonsTotal = 10\n    crayonsGiven = 2\n    crayonsLost = 4\n    crayonsLeft = crayonsTotal - crayonsGiven - crayonsLost\n    result = crayonsLeft\n\n    return result\n'}
11838003
1919


In [6]:
ds

DatasetDict({
    train: Dataset({
        features: ['question', 'code'],
        num_rows: 11838003
    })
    test: Dataset({
        features: ['question', 'code'],
        num_rows: 1919
    })
})

In [1]:
from collections import Counter
from math import ceil

test_questions = {
    q for q in test_df["question"].dropna()
    if q.strip()
}
matched_questions = set()
counts = Counter()
batch_size = 10_000

source = ds["train"].with_format(None).select_columns(["question", "code"])

for batch in tqdm(
    source.iter(batch_size=batch_size),
    total=ceil(len(source) / batch_size),
    desc="Полный проход train",
):
    counts["rows"] += len(batch["question"])

    for field in ["question", "code"]:
        for value in batch[field]:
            if value is None:
                counts[f"{field}_null"] += 1
            elif not value.strip():
                counts[f"{field}_empty_or_whitespace"] += 1

    for question in batch["question"]:
        if question in test_questions:
            counts["train_rows_matching_test_question"] += 1
            matched_questions.add(question)

counts["distinct_test_questions_found_in_train"] = len(matched_questions)

keys = [
    "rows",
    "question_null",
    "question_empty_or_whitespace",
    "code_null",
    "code_empty_or_whitespace",
    "train_rows_matching_test_question",
    "distinct_test_questions_found_in_train",
]
display(pd.Series({key: counts[key] for key in keys}).to_frame("count"))

NameError: name 'test_df' is not defined